# Step 6: Register the Model

**SageMaker Unified Studio Component**: Model Registry

**What you'll learn**: Register your model for versioning and governance

In [ ]:
import sagemaker
import os
from sagemaker.sklearn import SKLearnModel
from dotenv import load_dotenv

load_dotenv()
bucket_name = os.getenv('BUCKET_NAME')
role = os.getenv('EXECUTION_ROLE')

## Create Inference Script

In [ ]:
%%writefile inference.py
import joblib
import json
import numpy as np

def model_fn(model_dir):
    model = joblib.load(f"{model_dir}/model.pkl")
    return model

def input_fn(request_body, content_type):
    if content_type == 'application/json':
        data = json.loads(request_body)
        temp = data['temperature']
        room_temp = data['room_temp']
        temp_diff = temp - room_temp
        return np.array([[temp, temp_diff]])
    raise ValueError(f"Unsupported content type: {content_type}")

def predict_fn(input_data, model):
    prediction = model.predict(input_data)[0]
    probability = model.predict_proba(input_data)[0][1]
    return {'prediction': int(prediction), 'probability': float(probability)}

def output_fn(prediction, accept):
    return json.dumps(prediction), accept

## Register Model

In [ ]:
model_data = f's3://{bucket_name}/models/logistic_regression/model.tar.gz'

sklearn_model = SKLearnModel(
    model_data=model_data,
    role=role,
    entry_point='inference.py',
    framework_version='1.2-1',
    py_version='py3'
)

model_package = sklearn_model.register(
    content_types=['application/json'],
    response_types=['application/json'],
    inference_instances=['ml.t2.medium'],
    model_package_group_name='machine-overheat-models',
    approval_status='PendingManualApproval'
)

print(f"✓ Model registered: {model_package.model_package_arn}")